# Data (re)Preprocessing Rerun — Treating outside London as an additional MSOA, National deprivation frame - new geo harmonisation with 982 MSOAs - 20260625

**Changes from the both 20260615 and 20260622 version:**
- This version of data preprocessing extend flows with a "+1" MSOA for all outside London MSOAs. These flows treat London as either origin or destination.
- This version also adopts the new geo harmonisation with only 982 MSOAs.

**Everything else is identical to the 20260622 version.** 

**Purpose of 0622 version:**

The main analysis filters to London-internal flows only (both origin AND destination in London). This script extends the pipeline by also including flows where ONE endpoint is in London and the other is outside, treating the entire non-London area as a single synthetic "+1" MSOA.

However, previous preprocessing logic ranked MSOAs only within London. If we introduce outside London MSOAs, there is issue with their rankings. So in this file, all ~6,800 England MSOAs are ranked and assigned to deciles based on their IMD 2010 scores on a **national frame**.

Non-London areas are then collapsed into a single sythetic MSOA (`EXT_OUTSIDE`) whose decile is the population-weighted average of all non-London MSOAs' nationally-assigned deciles. This allows external flows to be classified as "wealthier inflow" or "poorer outflow" **within a consistent national hierarchy**, **testing whether including London-external flows changes the observed cascade-counter balance**.

**Design decisions:**

| Main analysis (existing) | This extension |
| :--- | :--- |
| Deciles: 983 London MSOAs only | Deciles: ~6,800 England MSOAs |
| Flows: London ↔ London | Flows: London ↔ anywhere |
| Frame: London-relative | Frame: National-relative |
| Purpose: Core analysis | Purpose: Sensitivity/extension |

**Input:**

census_od_2021_msoa.csv        (2021 MSOA-level OD, all E&W)
census_od_2011_oa.csv          (2011 OA-level OD, all E&W)
NSPCL_NOV22_UK_LU.csv          (postcode lookup: OA → LSOA → MSOA)
msoa_2011_to_2021_lookup.csv   (MSOA 2011 ↔ 2021 correspondence)
imd_2010.xls                   (IMD 2010 LSOA scores)
imd_2019.csv                   (IMD 2019, for mid-2015 LSOA populations)
ks101ew_lsoa_2011.csv          (2011 Census population by LSOA)
msoa_cascade_features_enriched_20260616.csv  (existing results to merge)

**Design:**
1. Aggregate IMD 2010 to all England MSOAs
2. Assign wealth deciles on the **national** distribution
3. Include London-external flows in the cascade computation
4. Compare results against the London-relative baseline


**Output:**

msoa_cascade_national_frame_20260625.csv

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
from scipy import stats
from pyprojroot import here

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — update paths to match your local setup
# ══════════════════════════════════════════════════════════════════════

ROOT = here()

DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Input files (same as main preprocessing)
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
census_od_2021_path = DATA_DIR / 'census_od_2021_msoa.csv'
census_od_2011_path = DATA_DIR / 'census_od_2011_oa.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'
existing_path       = OUTPUT_DIR / 'msoa_cascade_features_enriched_20260625.csv'

lookup_identification_path = DATA_DIR / 'MSOA_2011_to_2021_lookup_for_identification.csv'

# NEW: all-England KS101 population file
ks101_allengland_path = DATA_DIR / 'ks101ew_lsoa_2011_allengland.csv'

# Synthetic code for all non-London areas
EXTERNAL_CODE = 'EXT_OUTSIDE'

# IMD 2010 column names
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

## 1. Load raw data

In [3]:
# ---- IMD 2010 (LSOA level, all England) ----
imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()
print(f'IMD 2010: {len(imd_2010)} LSOAs')

# ---- KS101EW population (all England & Wales LSOAs) ----
ks101 = pd.read_csv(ks101_allengland_path)
# Detect LSOA code column: find column where values match E01/W01 pattern
_ks_code_col = next(
    c for c in ks101.columns
    if ks101[c].astype(str).str.match(r'^[EW]01\d{6}$').mean() > 0.9
)
_ks_pop_col = next(c for c in ks101.columns if 'all usual residents' in c.lower())
lsoa_pop = (ks101[[_ks_code_col, _ks_pop_col]]
            .rename(columns={_ks_code_col: 'lsoa11cd', _ks_pop_col: 'pop'}))
lsoa_pop['pop'] = pd.to_numeric(lsoa_pop['pop'], errors='coerce').fillna(0)
print(f'KS101EW: {len(lsoa_pop)} LSOAs, total pop = {lsoa_pop["pop"].sum():,.0f}')

# ---- NSPCL postcode lookup (LSOA → MSOA mapping) ----
nspcl = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
lsoa_to_msoa = nspcl[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_to_msoa = nspcl[['oa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_to_msoa_dict = dict(zip(oa_to_msoa['oa11cd'], oa_to_msoa['msoa11cd']))
print(f'LSOA→MSOA: {len(lsoa_to_msoa)} | OA→MSOA: {len(oa_to_msoa_dict):,}')

# ---- MSOA 2021→2011 correspondence ----
msoa_11_21 = pd.read_csv(msoa_lookup_path)
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()
unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']]
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))
print(f'MSOA 2021→2011 (1:1 only): {len(msoa21_to_11)}')

# ---- Existing London-only results (for comparison at the end) ----
existing = pd.read_csv(existing_path)
london_msoas = set(existing['msoa11cd'].unique())
print(f'London analysis MSOAs: {len(london_msoas)}')

IMD 2010: 32482 LSOAs
KS101EW: 34753 LSOAs, total pop = 56,075,912
LSOA→MSOA: 42621 | OA→MSOA: 232,044
MSOA 2021→2011 (1:1 only): 7080
London analysis MSOAs: 982


In [4]:
import sys; from pyprojroot import here; sys.path.insert(0, str(here()))
from geo_harmonise import build_harmonisation, build_full_remap
lookup_identification = pd.read_csv(lookup_identification_path)

h = build_harmonisation(lookup_identification, london_msoas)   # London (982) — for the merge collapse only
msoa21_to_11 = build_full_remap(lookup_identification)          # England-WIDE — keeps external codes real

lsoa_to_msoa['msoa11cd'] = lsoa_to_msoa['msoa11cd'].replace(h.collapse_2011)
oa_to_msoa_dict = {oa: h.collapse_2011.get(m, m) for oa, m in oa_to_msoa_dict.items()}
print(f'0622: full remap {len(msoa21_to_11)} codes | London frame {len(london_msoas)} | collapse {h.collapse_2011}')

0622: full remap 7264 codes | London frame 982 | collapse {'E02000190': 'E02000189'}


## 2. National IMD aggregation: LSOA -> MSOA (all England)

Aggregate IMD 2010 to ALL England MSOAs, then qcut into deciles

Same logic as the main preprocessing:
- Population-weighted mean of LSOA IMD scores per MSOA
- Re-rank MSOAs on the aggregated scores
- Assign deciles via qcut on the national distribution

In [5]:
# ---- 2a. Join IMD scores → MSOA geography → population weights ----
imd_lsoa = imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]].rename(
    columns={IMD_2010_LSOA_COL: 'lsoa11cd', IMD_2010_SCORE_COL: 'imd_score'})

# LSOA → MSOA
imd_with_msoa = pd.merge(imd_lsoa, lsoa_to_msoa, on='lsoa11cd', how='inner')
print(f'IMD LSOAs matched to MSOA: {len(imd_with_msoa)} / {len(imd_lsoa)}')

# Add population weights
imd_with_msoa = pd.merge(imd_with_msoa, lsoa_pop, on='lsoa11cd', how='left')
imd_with_msoa['pop'] = imd_with_msoa['pop'].fillna(0)

IMD LSOAs matched to MSOA: 31672 / 32482


In [6]:
# ---- 2b. Population-weighted mean per MSOA ----
# (Same fallback as main notebook: if all weights are zero, use simple mean)
def pop_weighted_mean(group):
    total_pop = group['pop'].sum()
    if total_pop > 0:
        return np.average(group['imd_score'], weights=group['pop'])
    else:
        return group['imd_score'].mean()

msoa_imd = (imd_with_msoa
            .groupby('msoa11cd')
            .apply(pop_weighted_mean, include_groups=False)
            .reset_index(name='IMD_2010_national'))

print(f'MSOAs with aggregated IMD: {len(msoa_imd)}')
print(f'  London MSOAs:     {msoa_imd["msoa11cd"].isin(london_msoas).sum()}')
print(f'  Non-London MSOAs: {(~msoa_imd["msoa11cd"].isin(london_msoas)).sum()}')


MSOAs with aggregated IMD: 6777
  London MSOAs:     982
  Non-London MSOAs: 5795


In [7]:
# ---- 2c. Assign national wealth deciles ----
# qcut splits into 10 bins by score. Higher IMD score = more deprived.
# pd.qcut with labels=False gives bin 0 = lowest scores = LEAST deprived.
# We want Decile 1 = most deprived, so: Decile = 11 - (bin + 1)
msoa_imd['Wealth_Decile_National'] = (
    11 - (pd.qcut(msoa_imd['IMD_2010_national'], 10, labels=False) + 1)
)

# Verify direction: D1 should have highest IMD scores (most deprived)
d1_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 1,
                         'IMD_2010_national'].mean()
d10_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 10,
                          'IMD_2010_national'].mean()
assert d1_score > d10_score, \
    f'Decile direction wrong: D1 mean={d1_score:.1f}, D10 mean={d10_score:.1f}'
print(f'\nDecile direction verified: D1 (most deprived, mean={d1_score:.1f}) > '
      f'D10 (least deprived, mean={d10_score:.1f})')



Decile direction verified: D1 (most deprived, mean=50.7) > D10 (least deprived, mean=5.7)


In [8]:
# ---- 2d. Summary: national decile composition ----
print(f'\nNational decile distribution:')
print(f'{"Decile":>6s} {"N_total":>8s} {"N_London":>9s} {"Mean IMD Score":>9s}')
for d in range(1, 11):
    mask = msoa_imd['Wealth_Decile_National'] == d
    n_total = mask.sum()
    n_london = (mask & msoa_imd['msoa11cd'].isin(london_msoas)).sum()
    mean_imd = msoa_imd.loc[mask, 'IMD_2010_national'].mean()
    print(f'  D{d:>2d}   {n_total:>7d}  {n_london:>8d}  {mean_imd:>9.2f}')



National decile distribution:
Decile  N_total  N_London Mean IMD Score
  D 1       678       107      50.73
  D 2       678       182      36.99
  D 3       677       148      29.63
  D 4       678       135      24.06
  D 5       677       107      19.72
  D 6       678        76      16.22
  D 7       678        74      13.40
  D 8       677        65      11.03
  D 9       678        52       8.74
  D10       678        36       5.73


> **Not the heavy tails on both ends as we expected? Specifically, there're more deprived MSOAs in London?**

**They key is that London is more deprived than the national average on the IMD, even though London is also obviously wealthy.**
- This is because of how the IMD is constructed.
- **The official IMD 2010 statistical release confirmed that London accounted for 19% of the most deprived 20% of areas natioanlly, which is aligned with our D1-D2 concentration.**

**When classifying flows within a London-only frame, a move from London D5 to London D6 crosses one decile boundary. But nationally, both of those areas might sits in national D3-D4, so the same physical move might not corss a national decile boundary at all, or might cross it differently.**

The national-frame analysis is:
- reclassifying some "cross-decile" London flows as "lateral" nationally
    - Because London's internal variation gets compressed into fewer national deciles.
- reclassifying some London external flows based on where the synthetic external outside-London MSOAs sits, which is identified as D6 in this file.(below)

In [9]:
# ---- 2e. External MSOA decile ----
# Population-weighted mean decile of all non-London MSOAs
non_london_imd = msoa_imd[~msoa_imd['msoa11cd'].isin(london_msoas)].copy()

# Get MSOA-level population for weighting
msoa_pop = (imd_with_msoa.groupby('msoa11cd')['pop'].sum()
            .reset_index(name='msoa_pop'))
non_london_imd = non_london_imd.merge(msoa_pop, on='msoa11cd', how='left')
non_london_imd['msoa_pop'] = non_london_imd['msoa_pop'].fillna(0)

# Safety check
total_ext_pop = non_london_imd['msoa_pop'].sum()
print(f'\nExternal MSOA calculation:')
print(f'  Non-London MSOAs: {len(non_london_imd)}, total pop: {total_ext_pop:,.0f}')
assert total_ext_pop > 0, 'External population is zero — check KS101 file coverage'

ext_weighted_decile = np.average(
    non_london_imd['Wealth_Decile_National'],
    weights=non_london_imd['msoa_pop'])
ext_decile = int(round(ext_weighted_decile))
print(f'  Weighted mean decile: {ext_weighted_decile:.2f} → assigned D{ext_decile}')


External MSOA calculation:
  Non-London MSOAs: 5795, total pop: 43,223,548
  Weighted mean decile: 5.67 → assigned D6


> **MSOAs of external flows are mainly in D6.**

In [10]:
# ---- 2f. Build wealth mapping (all MSOAs + external) ----
wealth_national = dict(zip(msoa_imd['msoa11cd'],
                            msoa_imd['Wealth_Decile_National']))
wealth_national[EXTERNAL_CODE] = ext_decile

In [11]:
# ---- 2g. Compare national vs London-relative deciles ----
print(f'\nLondon MSOA decile comparison (London-relative vs National):')
london_comparison = msoa_imd[msoa_imd['msoa11cd'].isin(london_msoas)].merge(
    existing[['msoa11cd', 'Wealth_Decile']], on='msoa11cd')

ct = pd.crosstab(london_comparison['Wealth_Decile'],
                 london_comparison['Wealth_Decile_National'], margins=True)
print(ct.to_string())

rho, p = stats.spearmanr(london_comparison['Wealth_Decile'],
                          london_comparison['Wealth_Decile_National'])
print(f'\nSpearman ρ = {rho:.4f}, p = {p:.2e}')


London MSOA decile comparison (London-relative vs National):
Wealth_Decile_National    1    2    3    4    5   6   7   8   9  10  All
Wealth_Decile                                                           
1                        99    0    0    0    0   0   0   0   0   0   99
2                         8   90    0    0    0   0   0   0   0   0   98
3                         0   92    6    0    0   0   0   0   0   0   98
4                         0    0   98    0    0   0   0   0   0   0   98
5                         0    0   44   54    0   0   0   0   0   0   98
6                         0    0    0   81   17   0   0   0   0   0   98
7                         0    0    0    0   90   8   0   0   0   0   98
8                         0    0    0    0    0  68  30   0   0   0   98
9                         0    0    0    0    0   0  44  54   0   0   98
10                        0    0    0    0    0   0   0  11  52  36   99
All                     107  182  148  135  107  76  74  65  5

## 3. Filter OD data: London-touching flows

Keep flows where at least one endpoint is a Lonson MSOA.
Non-London endpoints are recoded as `EXTERNAL_CODE`.
We exclude intra-MSOA moves.

In [12]:
# ---- 3a. 2021 Census OD (MSOA-level, 2021 codes) ----
print('\n--- 2021 ---')
ORIGIN_COL_2021 = 'Migrant MSOA one year ago code'
DEST_COL_2021   = 'Middle layer Super Output Areas code'
 
od_2021 = pd.read_csv(census_od_2021_path)
od_2021[ORIGIN_COL_2021] = od_2021[ORIGIN_COL_2021].astype(str).str.strip()
od_2021[DEST_COL_2021]   = od_2021[DEST_COL_2021].astype(str).str.strip()
 
# Detect count column
count_cols = [c for c in od_2021.columns
              if 'observation' in c.lower() or 'count' in c.lower()]
COUNT_COL_2021 = count_cols[0] if count_cols else od_2021.columns[-1]
print(f'Count column: {COUNT_COL_2021!r}')
print(f'Raw records: {len(od_2021):,}')


--- 2021 ---
Count column: 'Count'
Raw records: 1,490,726


In [13]:
# ── Step 1: Remove non-geography pseudo-codes ──────────────────────
# -8 = "Does not apply" (non-migrants); 999999999 = no fixed place;
# N99... = ONS "elsewhere" pseudo-codes.
# Without this, -8 rows (~53M non-migrant residents) get recoded as
# EXT_OUTSIDE inflows downstream.
# Ref: London-only notebook Section 4, classify_code_2021()
SPECIAL_2021 = {'-8', '999999999'}
 
is_special_origin = (od_2021[ORIGIN_COL_2021].isin(SPECIAL_2021) |
                     od_2021[ORIGIN_COL_2021].str.startswith('N99', na=False))
is_special_dest   = (od_2021[DEST_COL_2021].isin(SPECIAL_2021) |
                     od_2021[DEST_COL_2021].str.startswith('N99', na=False))
special_mask = is_special_origin | is_special_dest
 
print(f'Special-code rows removed:  {special_mask.sum():>10,} records  '
      f'{od_2021.loc[special_mask, COUNT_COL_2021].sum():>12,.0f} persons')
print(f'  of which -8 origin:       {is_special_origin.sum():>10,} records  '
      f'{od_2021.loc[is_special_origin, COUNT_COL_2021].sum():>12,.0f} persons')
 
od_2021 = od_2021[~special_mask].copy()
print(f'Records after filter: {len(od_2021):,}')

Special-code rows removed:      16,729 records    53,046,196 persons
  of which -8 origin:           16,729 records    53,046,196 persons
Records after filter: 1,473,997


In [14]:
# ── Step 2: Harmonise 2021→2011 codes ──────────────────────────────
od_2021['origin_msoa11'] = od_2021[ORIGIN_COL_2021].map(msoa21_to_11)
od_2021['dest_msoa11']   = od_2021[DEST_COL_2021].map(msoa21_to_11)

In [15]:
# ── Step 3: Classify London vs non-London ──────────────────────────
od_2021['o_ldn'] = od_2021['origin_msoa11'].isin(london_msoas)
od_2021['d_ldn'] = od_2021['dest_msoa11'].isin(london_msoas)

In [16]:
# Keep: at least one endpoint in London
lt_21 = od_2021[od_2021['o_ldn'] | od_2021['d_ldn']].copy()

In [17]:
# ── Step 4: Flow-weighted external decile diagnostic ───────────────
# Compute BEFORE collapsing non-London MSOAs, while individual codes
# are still available. Checks whether using a single D6 for
# EXT_OUTSIDE is a reasonable simplification.
ext_in_21  = lt_21[~lt_21['o_ldn'] & lt_21['d_ldn']].copy()
ext_out_21 = lt_21[lt_21['o_ldn'] & ~lt_21['d_ldn']].copy()
 
ext_in_21['o_dec']  = ext_in_21['origin_msoa11'].map(wealth_national)
ext_out_21['d_dec'] = ext_out_21['dest_msoa11'].map(wealth_national)
 
ext_in_21  = ext_in_21.dropna(subset=['o_dec'])
ext_out_21 = ext_out_21.dropna(subset=['d_dec'])
 
fw_in_21 = (np.average(ext_in_21['o_dec'], weights=ext_in_21[COUNT_COL_2021])
            if len(ext_in_21) > 0 and ext_in_21[COUNT_COL_2021].sum() > 0
            else np.nan)
fw_out_21 = (np.average(ext_out_21['d_dec'], weights=ext_out_21[COUNT_COL_2021])
             if len(ext_out_21) > 0 and ext_out_21[COUNT_COL_2021].sum() > 0
             else np.nan)
 
print(f'\n  Flow-weighted external decile (2021):')
print(f'    Population-weighted (all non-Ldn):  {ext_weighted_decile:.2f} → D{ext_decile}')
# print(f'    Inflow origins  (Ext→Ldn):          {fw_in_21:.2f} → D{int(round(fw_in_21))}  '
#       f'({ext_in_21[COUNT_COL_2021].sum():,.0f} persons)')
# print(f'    Outflow dests   (Ldn→Ext):          {fw_out_21:.2f} → D{int(round(fw_out_21))}  '
#       f'({ext_out_21[COUNT_COL_2021].sum():,.0f} persons)')
def _fmt(x): return f'{x:.2f} → D{int(round(x))}' if pd.notna(x) else 'n/a (no mapped external flows)'
print(f'    Inflow origins  (Ext→Ldn):          {_fmt(fw_in_21)}  ({ext_in_21[COUNT_COL_2021].sum():,.0f} persons)')
print(f'    Outflow dests   (Ldn→Ext):          {_fmt(fw_out_21)}  ({ext_out_21[COUNT_COL_2021].sum():,.0f} persons)')


  Flow-weighted external decile (2021):
    Population-weighted (all non-Ldn):  5.67 → D6
    Inflow origins  (Ext→Ldn):          6.28 → D6  (143,138 persons)
    Outflow dests   (Ldn→Ext):          6.40 → D6  (322,558 persons)


### Need to stressed in methodology:

We used 2 different ways of answering the question "what decile should we assign to `EXTERNAL_CODE`?"

**From section2, every MSOA in the data has a national-based decile. The below 2 methods evalute how to aggregate all outside London MSOAs' deciles into one.**

- **Population-weighted (used and kept)**

  Take all 5795 MSOAs and average their deciles weighted by how many people live in each one. The result 5.67 assigned as D6.

  This characterises the non-London population as a whole, and asking "if you pick a random person living outside London, what decile would they be in on average?"

  **The typical non-London resident lives in a D6-equivalent area.**

- **Flow-weighted (bottom 2 line test results)**
 
  Take only MSOAs that actually sent migrants to London and averages their deciles weighted by how many migrants each one sent Results is 6.28, assigned as D6 again.

  This checked if the population-weighted method is misleading. For example, if migrants came overwhelmingly from D9 areas, then calling external flows "D6" would systematically misclassify them.

**Reasons we need this comparison: They could have different results because migrants are not a random sample of the non-London population. As we discussed in the interpretation in the above cell, London's migration exchange is concerned with certain types of places, not uniformly with all of England.**

**But either method gives the same decile assignment as D6.**

In [18]:
# ── Step 5: Collapse non-London → EXTERNAL ─────────────────────────
lt_21.loc[~lt_21['o_ldn'], 'origin_msoa11'] = EXTERNAL_CODE
lt_21.loc[~lt_21['d_ldn'], 'dest_msoa11']   = EXTERNAL_CODE
 
# Drop intra-MSOA
lt_21 = lt_21[lt_21['origin_msoa11'] != lt_21['dest_msoa11']]

In [19]:
# ── Step 6: Aggregate ──────────────────────────────────────────────
flows_21 = (lt_21.groupby(['origin_msoa11', 'dest_msoa11'])[COUNT_COL_2021]
            .sum().reset_index().rename(columns={COUNT_COL_2021: 'count'}))
 
_i = flows_21[(flows_21['origin_msoa11'] != EXTERNAL_CODE) &
              (flows_21['dest_msoa11'] != EXTERNAL_CODE)]
_f = flows_21[flows_21['origin_msoa11'] == EXTERNAL_CODE]
_t = flows_21[flows_21['dest_msoa11'] == EXTERNAL_CODE]
print(f'\nInternal:  {len(_i):>7,} records  {_i["count"].sum():>10,.0f} persons')
print(f'Ext→Ldn:   {len(_f):>7,} records  {_f["count"].sum():>10,.0f} persons')
print(f'Ldn→Ext:   {len(_t):>7,} records  {_t["count"].sum():>10,.0f} persons')


Internal:  187,335 records     765,957 persons
Ext→Ldn:       982 records     154,172 persons
Ldn→Ext:       982 records     331,600 persons


In [20]:
# ---- 3b. 2011 Census OD (OA-level) ----
print('\n--- 2011 ---')
od_2011 = pd.read_csv(
    census_od_2011_path, header=None,
    names=['dest_oa', 'origin_oa', 'persons'],       # A=dest, B=origin, C=count
    dtype={'dest_oa': str, 'origin_oa': str, 'persons': int})
od_2011['origin_oa'] = od_2011['origin_oa'].str.strip()
od_2011['dest_oa']   = od_2011['dest_oa'].str.strip()
print(f'Raw OA records: {len(od_2011):,}')


--- 2011 ---
Raw OA records: 3,709,939


In [21]:
# ── Step 1: Remove OD-prefixed special codes ───────────────────────
# Parallel to 2021 filter. OD-prefixed rows are cross-border and
# no-fixed-origin pseudo-OAs. Small counts, but keeps logic clean.
# Ref: London-only notebook Section 4, classify_code_2011_oa()
is_special_2011 = (od_2011['origin_oa'].str.startswith('OD', na=False) |
                   od_2011['dest_oa'].str.startswith('OD', na=False))
print(f'Special-code rows removed:  {is_special_2011.sum():>10,} records  '
      f'{od_2011.loc[is_special_2011, "persons"].sum():>12,.0f} persons')
od_2011 = od_2011[~is_special_2011].copy()

Special-code rows removed:     116,320 records       612,488 persons


In [22]:
# ── Step 2: OA → MSOA ─────────────────────────────────────────────
od_2011['origin_msoa11'] = od_2011['origin_oa'].map(oa_to_msoa_dict)
od_2011['dest_msoa11']   = od_2011['dest_oa'].map(oa_to_msoa_dict)

In [23]:
# ── Step 3: Classify London vs non-London ──────────────────────────
od_2011['o_ldn'] = od_2011['origin_msoa11'].isin(london_msoas)
od_2011['d_ldn'] = od_2011['dest_msoa11'].isin(london_msoas)

In [24]:
# Keep: at least one endpoint in London
lt_11 = od_2011[od_2011['o_ldn'] | od_2011['d_ldn']].copy()

In [25]:
# ── Step 4: Flow-weighted external decile diagnostic ───────────────
ext_in_11  = lt_11[~lt_11['o_ldn'] & lt_11['d_ldn']].copy()
ext_out_11 = lt_11[lt_11['o_ldn'] & ~lt_11['d_ldn']].copy()
 
ext_in_11['o_dec']  = ext_in_11['origin_msoa11'].map(wealth_national)
ext_out_11['d_dec'] = ext_out_11['dest_msoa11'].map(wealth_national)
 
ext_in_11  = ext_in_11.dropna(subset=['o_dec'])
ext_out_11 = ext_out_11.dropna(subset=['d_dec'])
 
fw_in_11 = (np.average(ext_in_11['o_dec'], weights=ext_in_11['persons'])
            if len(ext_in_11) > 0 and ext_in_11['persons'].sum() > 0
            else np.nan)
fw_out_11 = (np.average(ext_out_11['d_dec'], weights=ext_out_11['persons'])
             if len(ext_out_11) > 0 and ext_out_11['persons'].sum() > 0
             else np.nan)
 
print(f'\n  Flow-weighted external decile (2011):')
print(f'    Population-weighted (all non-Ldn):  {ext_weighted_decile:.2f} → D{ext_decile}')
print(f'    Inflow origins  (Ext→Ldn):          {fw_in_11:.2f} → D{int(round(fw_in_11))}  '
      f'({ext_in_11["persons"].sum():,.0f} persons)')
print(f'    Outflow dests   (Ldn→Ext):          {fw_out_11:.2f} → D{int(round(fw_out_11))}  '
      f'({ext_out_11["persons"].sum():,.0f} persons)')


  Flow-weighted external decile (2011):
    Population-weighted (all non-Ldn):  5.67 → D6
    Inflow origins  (Ext→Ldn):          6.37 → D6  (170,474 persons)
    Outflow dests   (Ldn→Ext):          6.39 → D6  (212,248 persons)


In [26]:
# ── Step 5: Collapse non-London → EXTERNAL ─────────────────────────
lt_11.loc[~lt_11['o_ldn'], 'origin_msoa11'] = EXTERNAL_CODE
lt_11.loc[~lt_11['d_ldn'], 'dest_msoa11']   = EXTERNAL_CODE
 
# Drop intra-MSOA
lt_11 = lt_11[lt_11['origin_msoa11'] != lt_11['dest_msoa11']]

In [27]:
# ── Step 6: Aggregate to MSOA level ────────────────────────────────
flows_11 = (lt_11.groupby(['origin_msoa11', 'dest_msoa11'])['persons']
            .sum().reset_index().rename(columns={'persons': 'count'}))
 
_i = flows_11[(flows_11['origin_msoa11'] != EXTERNAL_CODE) &
              (flows_11['dest_msoa11'] != EXTERNAL_CODE)]
_f = flows_11[flows_11['origin_msoa11'] == EXTERNAL_CODE]
_t = flows_11[flows_11['dest_msoa11'] == EXTERNAL_CODE]
print(f'\nInternal:  {len(_i):>7,} records  {_i["count"].sum():>10,.0f} persons')
print(f'Ext→Ldn:   {len(_f):>7,} records  {_f["count"].sum():>10,.0f} persons')
print(f'Ldn→Ext:   {len(_t):>7,} records  {_t["count"].sum():>10,.0f} persons')


Internal:  173,346 records     768,324 persons
Ext→Ldn:       982 records     187,595 persons
Ldn→Ext:       982 records     219,221 persons


In [28]:
# ── Summary: flow-weighted vs population-weighted ──────────────────
print('\n' + '=' * 60)
print('EXTERNAL DECILE SUMMARY')
print('=' * 60)
print(f'  Population-weighted mean (all non-London):  D{ext_decile} ({ext_weighted_decile:.2f})')
print(f'  2011 inflow origins:   D{int(round(fw_in_11))} ({fw_in_11:.2f})   '
      f'diff from pop: {fw_in_11 - ext_weighted_decile:+.2f}')
print(f'  2011 outflow dests:    D{int(round(fw_out_11))} ({fw_out_11:.2f})   '
      f'diff from pop: {fw_out_11 - ext_weighted_decile:+.2f}')
print(f'  2021 inflow origins:   D{int(round(fw_in_21))} ({fw_in_21:.2f})   '
      f'diff from pop: {fw_in_21 - ext_weighted_decile:+.2f}')
print(f'  2021 outflow dests:    D{int(round(fw_out_21))} ({fw_out_21:.2f})   '
      f'diff from pop: {fw_out_21 - ext_weighted_decile:+.2f}')


EXTERNAL DECILE SUMMARY
  Population-weighted mean (all non-London):  D6 (5.67)
  2011 inflow origins:   D6 (6.37)   diff from pop: +0.70
  2011 outflow dests:    D6 (6.39)   diff from pop: +0.72
  2021 inflow origins:   D6 (6.28)   diff from pop: +0.62
  2021 outflow dests:    D6 (6.40)   diff from pop: +0.73


### Summary Interpretation:

The external synthetic MSOA was assigned a national wealth decile of D6 based on population-weighted mean of all non-London MSOAs.

Flow-weighted diagnostic confirmed this assignment with all values around to D6, validating the single-decile simplication across both census periods.

- Single D6 assignment is validated,
- The type of place London exchanges migrants with didn't meaningfully change between 2011 and 2021, even though the volume changes substantially. The structural geography of London's migration partnerships is durable.
- London exchanges with slightly wealthier-than-average England, not the "average England". (**all 4 flow-weighted deciles sit ~0.7 deciles above the population-weighted mean**)

## 4. Compute cascade metrics (national deciles, with external flows)

In [29]:
def compute_cascade(flow_df, wealth_map, year_label):
    """
    Per-London-MSOA cascade & counter-cascade metrics.
    Identical formulas to the main preprocessing, just different
    wealth_map (national deciles) and flow table (includes external).
    """
    df = flow_df.copy()
    df['o_dec'] = df['origin_msoa11'].map(wealth_map)
    df['d_dec'] = df['dest_msoa11'].map(wealth_map)

    # Drop unmapped
    before = len(df)
    df = df.dropna(subset=['o_dec', 'd_dec'])
    dropped = before - len(df)
    if dropped:
        print(f'  ⚠ {dropped} records dropped (unmapped decile)')
    df['o_dec'] = df['o_dec'].astype(int)
    df['d_dec'] = df['d_dec'].astype(int)

    # Summary
    '''
    `up` means destination decile > origin decile
    `down` means destination decile < origin decile
    `lat` means destination decile = origin decile
    '''
    total = df['count'].sum()
    up   = df.loc[df['d_dec'] > df['o_dec'], 'count'].sum()
    down = df.loc[df['d_dec'] < df['o_dec'], 'count'].sum()
    lat  = df.loc[df['d_dec'] == df['o_dec'], 'count'].sum()
    print(f'\n  {year_label}:')
    print(f'    Total:   {total:>10,.0f}')
    print(f'    Upward:  {up:>10,.0f} ({up/total*100:.1f}%)')
    print(f'    Down:    {down:>10,.0f} ({down/total*100:.1f}%)')
    print(f'    Lateral: {lat:>10,.0f} ({lat/total*100:.1f}%)')

    # ---- Per-London-MSOA metrics ----
    idx = sorted(london_msoas)

    # Cascade: wealthier in, poorer out
    inflow_w  = (df[df['o_dec'] > df['d_dec']].groupby('dest_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))
    outflow_p = (df[df['d_dec'] < df['o_dec']].groupby('origin_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))

    # Counter-cascade: wealthier out, poorer in
    outflow_w = (df[df['d_dec'] > df['o_dec']].groupby('origin_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))
    inflow_p  = (df[df['o_dec'] < df['d_dec']].groupby('dest_msoa11')['count']
                 .sum().reindex(idx, fill_value=0))

    # Totals
    total_in  = df.groupby('dest_msoa11')['count'].sum().reindex(idx, fill_value=0)
    total_out = df.groupby('origin_msoa11')['count'].sum().reindex(idx, fill_value=0)

    # External-specific
    ext_in  = (df[df['origin_msoa11'] == EXTERNAL_CODE]
               .groupby('dest_msoa11')['count'].sum().reindex(idx, fill_value=0))
    ext_out = (df[df['dest_msoa11'] == EXTERNAL_CODE]
               .groupby('origin_msoa11')['count'].sum().reindex(idx, fill_value=0))

    # Assemble
    r = pd.DataFrame({'msoa11cd': idx})
    r = r.set_index('msoa11cd')
    r['Inflow_Wealthier']  = inflow_w.values
    r['Outflow_Poorer']    = outflow_p.values
    r['Outflow_Wealthier'] = outflow_w.values
    r['Inflow_Poorer']     = inflow_p.values
    r['Total_Inflow']      = total_in.values
    r['Total_Outflow']     = total_out.values
    r['Ext_Inflow']        = ext_in.values
    r['Ext_Outflow']       = ext_out.values

    # Derived metrics (same formulas as main notebook)
    r['Total_Migration'] = r['Total_Inflow'] + r['Total_Outflow']
    r['CFI_Churn']   = r['Inflow_Wealthier'] + r['Outflow_Poorer']
    r['CFI_Rate']    = np.where(r['Total_Migration'] > 0,
        (r['Inflow_Wealthier'] * r['Outflow_Poorer']) / r['Total_Migration'], 0)
    r['Net_Cascade'] = r['Inflow_Wealthier'] - r['Outflow_Poorer']
    r['Pct_Inflow_Wealthier'] = np.where(r['Total_Inflow'] > 0,
        r['Inflow_Wealthier'] / r['Total_Inflow'] * 100, 0)

    r['Counter_Churn'] = r['Outflow_Wealthier'] + r['Inflow_Poorer']
    r['Counter_Rate']  = np.where(r['Total_Migration'] > 0,
        (r['Outflow_Wealthier'] * r['Inflow_Poorer']) / r['Total_Migration'], 0)
    r['Net_Counter']   = r['Outflow_Wealthier'] - r['Inflow_Poorer']

    # Cascade Dominance
    total_cross = r['CFI_Churn'] + r['Counter_Churn']
    r['Cascade_Dominance'] = np.where(total_cross > 0,
        r['CFI_Churn'] / total_cross, 0.5)

    r['Ext_Net'] = r['Ext_Inflow'] - r['Ext_Outflow']
    return r


In [30]:
# Compute both periods
print('\n--- 2011 ---')
nat_11 = compute_cascade(flows_11, wealth_national, '2011 national frame')
print('\n--- 2021 ---')
nat_21 = compute_cascade(flows_21, wealth_national, '2021 national frame')

# Add _nat suffix + year suffix
nat_11.columns = [f'{c}_nat_11' for c in nat_11.columns]
nat_21.columns = [f'{c}_nat_21' for c in nat_21.columns]


--- 2011 ---

  2011 national frame:
    Total:    1,175,140
    Upward:     493,105 (42.0%)
    Down:       487,394 (41.5%)
    Lateral:    194,641 (16.6%)

--- 2021 ---

  2021 national frame:
    Total:    1,251,729
    Upward:     580,241 (46.4%)
    Down:       478,578 (38.2%)
    Lateral:    192,910 (15.4%)


## 5. Merge and Compare London-only vs. National frame

In [31]:
# Add national decile
nat_deciles = msoa_imd[msoa_imd['msoa11cd'].isin(london_msoas)][
    ['msoa11cd', 'Wealth_Decile_National']]

result = existing.merge(nat_deciles, on='msoa11cd', how='left')
result = result.merge(nat_11, left_on='msoa11cd', right_index=True, how='left')
result = result.merge(nat_21, left_on='msoa11cd', right_index=True, how='left')
print(f'Final shape: {result.shape}')

Final shape: (982, 98)


In [32]:
# ---- Cascade Dominance comparison ----
print('\n' + '-' * 50)
print('CASCADE DOMINANCE COMPARISON')
print('-' * 50)

for yr in ['11', '21']:
    dom_ldn = f'Cascade_Dominance_{yr}'
    dom_nat = f'Cascade_Dominance_nat_{yr}'

    if dom_ldn in result.columns and dom_nat in result.columns:
        ldn_mean   = result[dom_ldn].mean()
        nat_mean   = result[dom_nat].mean()
        ldn_below  = (result[dom_ldn] < 0.5).mean() * 100
        nat_below  = (result[dom_nat] < 0.5).mean() * 100

        print(f'\n20{yr}:')
        print(f'  London-only:   mean = {ldn_mean:.4f}  '
              f'({ldn_below:.1f}% of MSOAs < 0.5)')
        print(f'  National+ext:  mean = {nat_mean:.4f}  '
              f'({nat_below:.1f}% of MSOAs < 0.5)')
        print(f'  Shift:         {nat_mean - ldn_mean:+.4f}')

        rho, p = stats.spearmanr(result[dom_ldn], result[dom_nat])
        print(f'  Spearman ρ:    {rho:.4f} (p={p:.2e})')



--------------------------------------------------
CASCADE DOMINANCE COMPARISON
--------------------------------------------------

2011:
  London-only:   mean = 0.4796  (67.6% of MSOAs < 0.5)
  National+ext:  mean = 0.4862  (59.2% of MSOAs < 0.5)
  Shift:         +0.0067
  Spearman ρ:    0.5489 (p=2.28e-78)

2021:
  London-only:   mean = 0.4644  (80.9% of MSOAs < 0.5)
  National+ext:  mean = 0.4551  (73.6% of MSOAs < 0.5)
  Shift:         -0.0093
  Spearman ρ:    0.3018 (p=4.01e-22)


In [33]:
# ---- Quick check: share of London MSOAs nationally below D6 ----
ldn_nat = msoa_imd.loc[msoa_imd['msoa11cd'].isin(london_msoas),
                       'Wealth_Decile_National'].dropna()
below_d6 = (ldn_nat < 6)
print(f"London MSOAs below D6 nationally: "
      f"{below_d6.sum()}/{len(ldn_nat)} = {below_d6.mean():.1%}")

London MSOAs below D6 nationally: 679/982 = 69.1%


In [34]:
# ---- CFI Churn comparison ----
print('\n' + '-' * 50)
print('CFI CHURN COMPARISON')
print('-' * 50)
for yr in ['11', '21']:
    c_ldn = f'CFI_Churn_{yr}'
    c_nat = f'CFI_Churn_nat_{yr}'
    if c_ldn in result.columns and c_nat in result.columns:
        print(f'\n20{yr}:')
        print(f'  London-only mean: {result[c_ldn].mean():>10.1f}')
        print(f'  National mean:    {result[c_nat].mean():>10.1f}')
        print(f'  Increase:         {result[c_nat].mean() - result[c_ldn].mean():>+10.1f}')



--------------------------------------------------
CFI CHURN COMPARISON
--------------------------------------------------

2011:
  London-only mean:      630.1
  National mean:         797.9
  Increase:             +167.8

2021:
  London-only mean:      614.2
  National mean:         781.0
  Increase:             +166.8


In [35]:
# ---- Counter Churn comparison ----
print('\n' + '-' * 50)
print('COUNTER CHURN COMPARISON')
print('-' * 50)
for yr in ['11', '21']:
    co_ldn = f'Counter_Churn_{yr}'
    co_nat = f'Counter_Churn_nat_{yr}'
    if co_ldn in result.columns and co_nat in result.columns:
        print(f'\n20{yr}:')
        print(f'  London-only mean: {result[co_ldn].mean():>10.1f}')
        print(f'  National mean:    {result[co_nat].mean():>10.1f}')
        print(f'  Increase:         {result[co_nat].mean() - result[co_ldn].mean():>+10.1f}')



--------------------------------------------------
COUNTER CHURN COMPARISON
--------------------------------------------------

2011:
  London-only mean:      666.7
  National mean:         819.5
  Increase:             +152.8

2021:
  London-only mean:      687.0
  National mean:         919.6
  Increase:             +232.7


In [36]:
# ---- External flow volumes ----
print('\n' + '-' * 50)
print('EXTERNAL FLOW VOLUMES')
print('-' * 50)
for yr in ['11', '21']:
    ei = f'Ext_Inflow_nat_{yr}'
    eo = f'Ext_Outflow_nat_{yr}'
    en = f'Ext_Net_nat_{yr}'
    if ei in result.columns:
        print(f'\n20{yr}:')
        print(f'  Mean inflow from outside:  {result[ei].mean():>8.1f}')
        print(f'  Mean outflow to outside:   {result[eo].mean():>8.1f}')
        print(f'  Mean net external:         {result[en].mean():>+8.1f}')


--------------------------------------------------
EXTERNAL FLOW VOLUMES
--------------------------------------------------

2011:
  Mean inflow from outside:     191.0
  Mean outflow to outside:      223.2
  Mean net external:            -32.2

2021:
  Mean inflow from outside:     157.0
  Mean outflow to outside:      337.7
  Mean net external:           -180.7


### Churn and flow volume interpretation:

Both churns increase because of external flows added to the system.

**The key finding is the asymmetry in increase for each flow. The national-frame extension reveals that London's boundary effects are not temporally neutral. In 2011, external flows were roughly balanced between cascade and counter directions, leaving the London-only story essentially intact. By 2021, pandemic-era outmigration created a strong counter asymmetry in external flows, meaning the London-only analysis understates the counter-cascade intensification. As we dicussed above, the finding that counter-cascade dominance strengthened between censuses, holds even more firmly once boundary effects are accounted for.**
- In 2011,
    - cascade churn increase and counter churn increase are roughly balanced, as external flows fed both directions almost euqally.
    - This is why cascade dominance barely shifted towards *cascade* direction.

- In 2021,
    - The counter side absorbed significantly more of the external flows than cascade (cascade grew by 166 but counter grew by 232).
    - This is why cascade dominance shifted toward *counter*, despite both churns increasing.

## 6. Export

In [37]:
out_path = OUTPUT_DIR / f'msoa_cascade_national_frame_20260625.csv'
result.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')
print(f'  Shape: {result.shape}')



Saved: /Users/xing/Desktop/CASA/dissertation/outputs/msoa_cascade_national_frame_20260625.csv
  Shape: (982, 98)


### Additional check - Migration Distance

In [38]:
# ── Additional check: flow-weighted move distance by hierarchy direction ──
# Private diagnostic (viva preparation), not for thesis main text.
# London-internal flows only: EXTERNAL node has no meaningful centroid.
import geopandas as gpd

gdf_geo = gpd.read_file(ROOT / 'data/london_msoa_2011.geojson').to_crs(27700)

print(gdf_geo.columns.tolist())   # see what's there

# auto-detect: the ID column is the one full of 'E02...' codes
id_col = next(c for c in gdf_geo.columns if c != 'geometry'
              and gdf_geo[c].astype(str).str.startswith('E02').mean() > 0.9)
print(f'Using ID column: {id_col}')

cent = gdf_geo.set_index(id_col).geometry.centroid

xy = pd.DataFrame({'x': cent.x, 'y': cent.y})

wealth_london = dict(zip(existing['msoa11cd'], existing['Wealth_Decile']))

def distance_check(flows, count_col, ladder, label):
    f = flows[(flows['origin_msoa11'] != EXTERNAL_CODE) &
              (flows['dest_msoa11']   != EXTERNAL_CODE)].copy()
    f = f.join(xy.add_prefix('o_'), on='origin_msoa11')
    f = f.join(xy.add_prefix('d_'), on='dest_msoa11')
    f['dist_km'] = np.hypot(f['o_x'] - f['d_x'], f['o_y'] - f['d_y']) / 1000
    f['o_dec'] = f['origin_msoa11'].map(ladder)
    f['d_dec'] = f['dest_msoa11'].map(ladder)
    f['direction'] = np.select(
        [f['d_dec'] < f['o_dec'], f['d_dec'] > f['o_dec']],
        ['downward (cascade-consistent)', 'upward (counter-consistent)'],
        default='lateral')

    def wstats(g):
        s = g.sort_values('dist_km')
        cum = s[count_col].cumsum() / s[count_col].sum()
        return pd.Series({'w_median_km': s.loc[cum >= 0.5, 'dist_km'].iloc[0],
                          'w_mean_km': np.average(s['dist_km'], weights=s[count_col]),
                          'persons': s[count_col].sum()})

    out = f.groupby('direction').apply(wstats)
    print(f'\n=== {label} ===\n', out.round(2))
    return out

d11 = distance_check(flows_11, 'count', wealth_london, '2011, London ladder')
d21 = distance_check(flows_21, 'count', wealth_london, '2021, London ladder')

['MSOA11CD', 'MSOA11NM', 'geometry']
Using ID column: MSOA11CD

=== 2011, London ladder ===
                                w_median_km  w_mean_km   persons
direction                                                      
downward (cascade-consistent)         3.57       5.62  309399.0
lateral                               2.52       4.07  131551.0
upward (counter-consistent)           3.93       5.93  327374.0

=== 2021, London ladder ===
                                w_median_km  w_mean_km   persons
direction                                                      
downward (cascade-consistent)         4.05       6.09  301568.0
lateral                               2.78       4.47  127077.0
upward (counter-consistent)           4.42       6.41  337312.0


In [39]:
flow_codes = set(flows_21['origin_msoa11']) | set(flows_21['dest_msoa11'])
flow_codes.discard(EXTERNAL_CODE)
missing = flow_codes - set(cent.index)
print(f'{len(missing)} flow MSOAs missing from geojson:', sorted(missing)[:5])

0 flow MSOAs missing from geojson: []
